# RAG Pro Implementation — Learning Notebook

This notebook turns the advanced RAG techniques from the Day 5 material into an implementation-focused learning path.

The goal is to understand **why each technique exists, how it changes the pipeline, and how to experiment with it**.

Covered:
1. Chunking R&D
2. Encoder / embedding R&D
3. Prompt improvement
4. Document pre-processing
5. Query rewriting
6. Query expansion
7. Re-ranking
8. Hierarchical RAG
9. Graph RAG
10. Agentic RAG

## How to use this notebook

Run the sections in order.

For experiments, change **one thing at a time** and evaluate the same fixed test set.

```text
Baseline
   ↓
Chunking A → Chunking B
   ↓
Embedding A → Embedding B
   ↓
Query rewriting
   ↓
Query expansion
   ↓
Re-ranking
```

Keep the same documents, questions, `k`, and evaluation set whenever possible. This makes the results comparable.

## 0. Setup

This version uses **Hugging Face embeddings locally**, so embedding experiments do not consume Gemini embedding quota.

The LLM layer uses LiteLLM. Set `RAG_LLM_MODEL` in `.env` to the model/provider you want to use.

The notebook follows the Day 5 architecture of native Chroma + structured LLM outputs, while adding the other advanced RAG techniques.

In [9]:
# Install once in your project environment if needed:
# pip install chromadb litellm pydantic python-dotenv
# pip install langchain-text-splitters langchain-huggingface
# pip install sentence-transformers networkx scikit-learn matplotlib

import os
import glob
import json
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from litellm import completion
from chromadb import PersistentClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)

LLM_MODEL = os.getenv("RAG_LLM_MODEL", "gemini/gemini-2.5-flash-lite")

DB_NAME = Path("pro_rag_db")
KNOWLEDGE_BASE = Path("knowledge-base")
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

RETRIEVAL_K = 20
FINAL_K = 10

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

print("LLM:", LLM_MODEL)
print("Embedding:", EMBEDDING_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9216.83it/s]


LLM: gemini/gemini-2.5-flash-lite
Embedding: BAAI/bge-small-en-v1.5


## 1. Load the knowledge base

Walk through the Markdown files and preserve useful metadata. Metadata can later be used for filtering, citations, debugging, hierarchical retrieval, and graph links.

In [10]:
def fetch_documents():
    documents = []

    for folder in KNOWLEDGE_BASE.iterdir():
        if not folder.is_dir():
            continue

        doc_type = folder.name

        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({
                    "type": doc_type,
                    "source": file.as_posix(),
                    "text": f.read(),
                })

    print(f"Loaded {len(documents)} documents")
    return documents


documents = fetch_documents()

Loaded 76 documents


## 2. Baseline chunking

Chunking decides **what unit gets embedded and retrieved**.

Small chunks can lose context. Large chunks can contain unrelated information. There is no universally best size, so start with a baseline and evaluate alternatives.

In [11]:
def create_chunks_baseline(documents, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = []

    for doc in documents:
        pieces = splitter.split_text(doc["text"])

        for i, piece in enumerate(pieces):
            chunks.append({
                "page_content": piece,
                "metadata": {
                    "source": doc["source"],
                    "type": doc["type"],
                    "chunk_id": i,
                },
            })

    return chunks


chunks = create_chunks_baseline(
    documents,
    chunk_size=1000,
    chunk_overlap=150,
)

print("Chunks:", len(chunks))

Chunks: 399


## 3. Chunking R&D

Chunking R&D means **experimenting instead of guessing**.

Try:
- different chunk sizes
- different overlaps
- Markdown-aware splitting
- semantic splitting
- LLM-based chunking
- parent-child chunks

The Day 5 implementation uses an LLM to decide boundaries and produce a headline, summary, and original text for each chunk.

In [12]:
class LLMChunk(BaseModel):
    headline: str = Field(description="A short heading describing the chunk")
    summary: str = Field(description="A short factual summary useful for retrieval")
    original_text: str = Field(description="The original text, unchanged")


class LLMChunks(BaseModel):
    chunks: list[LLMChunk]


def llm_chunk_document(document):
    prompt = f"""
Split this document into overlapping chunks for a company knowledge base.

Document type: {document["type"]}
Source: {document["source"]}

Keep the complete document represented across the chunks.
Choose boundaries that keep related information together.

For every chunk return:
1. headline
2. short summary
3. original_text exactly as it appears

Document:
{document["text"]}
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=LLMChunks,
    )

    parsed = LLMChunks.model_validate_json(
        response.choices[0].message.content
    )

    return [
        {
            "page_content": (
                item.headline
                + "\n\n"
                + item.summary
                + "\n\n"
                + item.original_text
            ),
            "metadata": {
                "source": document["source"],
                "type": document["type"],
            },
        }
        for item in parsed.chunks
    ]


#Example:
llm_chunks = []

for doc in documents[0:3]:
    llm_chunks.extend(llm_chunk_document(doc))
    
print(documents[0])
print(len(llm_chunks))
for i in llm_chunks:
    print(i)
llm_chunks

{'type': 'products', 'source': 'knowledge-base/products/Rellm.md', 'text': "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integra

[{'page_content': 'Rellm: AI-Powered Enterprise Reinsurance Solution\n\nRellm is an AI-driven enterprise reinsurance product by Insurellm that transforms risk management, decision-making, and operational efficiency for reinsurance companies.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.',
  'metadata': {'source': 'knowledge-base/products/Rellm.md',
   'type': 'products'}},
 {'page_content': "Rellm Features: AI Analytics, Integratio

## 4. Encoder / embedding R&D

The encoder converts text into vectors. Keep everything else fixed and swap only the embedding model.

Good local experiments:
- `sentence-transformers/all-MiniLM-L6-v2` — lightweight baseline
- `BAAI/bge-small-en-v1.5` — strong lightweight English retrieval
- `sentence-transformers/all-mpnet-base-v2` — larger general-purpose model
- `intfloat/e5-small-v2` — efficient retrieval model

**Important:** changing the embedding model means rebuilding the vector collection. Do not mix vectors from different embedding spaces.

In [13]:
# Change only this value for an embedding experiment.

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Using:", EMBEDDING_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6827.55it/s]


Using: sentence-transformers/all-MiniLM-L6-v2


## 5. Build the vector store

Baseline indexing:

```text
documents → chunks → embeddings → Chroma
```

The source implementation stores metadata alongside every vector so the source can be shown later.

In [14]:
def build_vector_store(chunks, db_path=DB_NAME, collection_name="docs"):
    client = PersistentClient(path=str(db_path))

    existing = [c.name for c in client.list_collections()]

    if collection_name in existing:
        client.delete_collection(collection_name)

    collection = client.get_or_create_collection(collection_name)

    texts = [c["page_content"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]

    vectors = embeddings.embed_documents(texts)

    collection.add(
        ids=[str(i) for i in range(len(chunks))],
        embeddings=vectors,
        documents=texts,
        metadatas=metadatas,
    )

    print("Vectorstore created:", collection.count())
    return client, collection


# Run when ready:
client, collection = build_vector_store(llm_chunks)

Vectorstore created: 14


## 6. Baseline retrieval

Separate retrieval from generation so retrieval can be evaluated independently.

```text
question
   ↓
query embedding
   ↓
vector similarity
   ↓
top K chunks
```

The Day 5 implementation retrieves a larger candidate set first and later reduces it to a final context set.

In [15]:
def fetch_context_unranked(question, collection, k=RETRIEVAL_K):
    query_vector = embeddings.embed_query(question)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
    )

    chunks = []

    for document, metadata in zip(
        results["documents"][0],
        results["metadatas"][0],
    ):
        chunks.append({
            "page_content": document,
            "metadata": metadata,
        })

    return chunks


# Example:
retrieved = fetch_context_unranked(
    "About rellm",
    collection,
)
print("Retrieved:", len(retrieved))
for i in retrieved:
    print(i)

Retrieved: 14
{'page_content': "Rellm Features: AI Analytics, Integrations, Risk Assessment, Dashboard, Compliance, and Portals\n\nRellm offers AI-driven analytics for predictive insights, seamless integrations with existing systems, a comprehensive risk assessment module, a customizable dashboard, regulatory compliance tools, and dedicated portals for clients and brokers.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integrations\nRellm's architecture is designed for effortless integration with existing systems. Whether it's policy management, claims processing, or financial reporting, Rellm connects seamlessly with diverse data sources to create a unified ecosystem.\n\n### Risk Assessment Module\nThe comprehensive risk assess

# Advanced Retrieval Techniques

The following sections change different parts of the retrieval pipeline. Keep the baseline available so every improvement can be compared against it.

## 7. Improve Prompts

A good RAG prompt defines:
- the assistant's role
- how retrieved context should be used
- what to do when context is insufficient
- current date when relevant
- conversation history when relevant
- answer constraints such as accuracy, relevance, and completeness

Better prompting helps generation, but it cannot fix fundamentally bad retrieval.

In [16]:
SYSTEM_PROMPT = """
You are a knowledgeable and precise assistant.

Answer the user's question using the provided knowledge-base context.

Rules:
- Use the retrieved context as the primary factual source.
- Do not invent facts that are not supported by the context.
- If the context is insufficient, say that you do not know.
- Answer the exact question and avoid unnecessary information.
- Use conversation history when needed.
- Preserve source information when useful.

Retrieved context:
{context}
"""


def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Source: {chunk['metadata'].get('source', 'unknown')}\n"
        f"{chunk['page_content']}"
        for chunk in chunks
    )

    return (
        [
            {
                "role": "system",
                "content": SYSTEM_PROMPT.format(context=context),
            }
        ]
        + history
        + [{"role": "user", "content": question}]
    )

## 8. Document pre-processing

Document pre-processing changes the representation **before embedding**.

The Day 5 approach uses an LLM to create:

```text
headline + summary + original text
```

The headline and summary can improve semantic matching while the original text preserves the actual evidence.

In [17]:
def preprocess_chunk(chunk):
    prompt = f"""
Improve this knowledge-base chunk for retrieval.

Return:
- a short headline
- a concise factual summary
- the original text exactly as provided

Do not add facts.

Chunk:
{chunk["page_content"]}
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=LLMChunk,
    )

    item = LLMChunk.model_validate_json(
        response.choices[0].message.content
    )

    return {
        "page_content": (
            item.headline
            + "\n\n"
            + item.summary
            + "\n\n"
            + item.original_text
        ),
        "metadata": chunk["metadata"],
    }


# Example:
processed = preprocess_chunk(chunks[0])
print(processed["page_content"])

Rellm: AI-Powered Enterprise Reinsurance Solution

Rellm is an AI-powered enterprise reinsurance product by Insurellm that transforms operations by enhancing risk management, decision-making, and operational efficiencies through advanced analytics and seamless integrations.

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk e

## 9. Query rewriting

Users often ask questions conversationally or vaguely.

Query rewriting uses an LLM to turn the user's question, optionally using history, into a short search query containing the details needed for retrieval.

In [19]:
def rewrite_query(question, history=None):
    history = history or []

    prompt = f"""
You are preparing a search query for a company knowledge base.

Conversation history:
{history}

Current question:
{question}

Rewrite the question into one short, specific search query.
Resolve references from the conversation when possible.
Return only the search query.
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.choices[0].message.content.strip()


# Example:
rewritten = rewrite_query(
    "What about the guy who went there?",
    [{"role": "user", "content": "Who went to Manchester University?"}]
)
print(rewritten)

"guy who went to Manchester University"


## 10. Query expansion

**Rewriting = one better query.**

**Expansion = multiple related queries.**

Expansion helps when different terminology or perspectives may retrieve different relevant chunks. Retrieve for each query, merge, and deduplicate.

In [21]:
class QuerySet(BaseModel):
    queries: list[str] = Field(
        description="Different search queries representing the same question"
    )


def expand_query(question, number=3):
    prompt = f"""
Create {number} different search queries for the same knowledge-base question.

Question:
{question}

Use different wording or information angles.
Return only the queries.
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=QuerySet,
    )

    return QuerySet.model_validate_json(
        response.choices[0].message.content
    ).queries


def merge_unique_chunks(chunk_lists):
    merged = []
    seen = set()

    for result_set in chunk_lists:
        for chunk in result_set:
            key = (
                chunk["metadata"].get("source"),
                chunk["page_content"],
            )

            if key not in seen:
                seen.add(key)
                merged.append(chunk)

    return merged


def fetch_context_with_expansion(question, collection, k=10):
    queries = expand_query(question)

    result_sets = [
        fetch_context_unranked(q, collection, k=k)
        for q in queries
    ]

    return merge_unique_chunks(result_sets)


# Example:
print(expand_query("Who won the IIOTY award?"))

['IIOTY award winner', 'Who received the IIOTY award?', 'Recipients of the IIOTY award']


## 11. Re-ranking

Vector search produces candidates. A second relevance step can improve the ordering.

```text
Query
  ↓
Vector retrieval: top 20
  ↓
Re-ranker
  ↓
Best 10
```

The Day 5 implementation asks an LLM to return chunk IDs ordered from most relevant to least relevant and validates that response with Pydantic.

In [22]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Chunk IDs ordered from most relevant to least relevant"
    )


def rerank(question, chunks):
    if not chunks:
        return []

    chunk_text = "\n\n".join(
        f"# CHUNK {i + 1}\n{chunk['page_content']}"
        for i, chunk in enumerate(chunks)
    )

    prompt = f"""
You are a document re-ranker.

Question:
{question}

Rank every provided chunk from most relevant to least relevant.
Return every chunk ID exactly once.

Chunks:
{chunk_text}
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=RankOrder,
    )

    order = RankOrder.model_validate_json(
        response.choices[0].message.content
    ).order

    valid = [i for i in order if 1 <= i <= len(chunks)]

    return [chunks[i - 1] for i in valid]


def fetch_context_reranked(question, collection, retrieval_k=20, final_k=10):
    candidates = fetch_context_unranked(
        question,
        collection,
        k=retrieval_k,
    )

    ranked = rerank(question, candidates)

    return ranked[:final_k]

## 12. Combine rewrite + expansion + re-ranking

A strong hybrid retrieval path can be:

```text
Original question
      ↓
Rewrite
      ↓
Expansion
      ↓
Retrieve each query
      ↓
Merge + deduplicate
      ↓
Re-rank
      ↓
Final K
```

This adds latency and LLM calls, so evaluate whether the improvement is worth the cost.

In [23]:
def advanced_retrieval(question, collection, retrieval_k=20, final_k=10):
    rewritten = rewrite_query(question)

    queries = [question, rewritten]

    for q in expand_query(question, number=2):
        if q not in queries:
            queries.append(q)

    result_sets = [
        fetch_context_unranked(
            q,
            collection,
            k=retrieval_k,
        )
        for q in queries
    ]

    candidates = merge_unique_chunks(result_sets)

    ranked = rerank(question, candidates)

    return ranked[:final_k]


# Example:
context = advanced_retrieval(
    "Who went there and what did they do?",
    collection,
)

# 13. Hierarchical RAG

Hierarchical RAG stores information at multiple levels.

```text
Document
   ↓
Document summary
   ↓
Section summary
   ↓
Detailed chunks
```

Retrieve at a coarse level first, then drill down into detailed chunks.

This is useful for large documents where searching every small chunk independently loses document structure.

In [24]:
class Summary(BaseModel):
    summary: str
    source: str


def summarize_document(document):
    prompt = f"""
Summarize this document for a hierarchical knowledge base.

Keep:
- important entities
- important facts
- topics covered
- important relationships

Document:
{document["text"]}
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )

    return Summary(
        summary=response.choices[0].message.content,
        source=document["source"],
    )


# Example:
summary = summarize_document(documents[0])
print(summary.summary)

Here's a summary of the document for a hierarchical knowledge base:

**Top Level Entity:** Rellm

**Description:** An AI-powered enterprise reinsurance solution developed by Insurellm.

**Key Features:**

*   **AI-Driven Analytics:**
    *   Utilizes AI algorithms for predictive insights into risk exposures.
    *   Enables forecasting of trends.
    *   Provides real-time data analysis and actionable intelligence.
*   **Seamless Integrations:**
    *   Designed for effortless integration with existing systems (policy management, claims processing, financial reporting).
    *   Connects with diverse data sources to create a unified ecosystem.
*   **Risk Assessment Module:**
    *   Evaluates risk profiles accurately.
    *   Leverages historical data and advanced modeling techniques.
    *   Provides insights into potential liabilities and expected outcomes.
*   **Customizable Dashboard:**
    *   Presents key metrics and performance indicators.
    *   Intuitive interface.
    *   All

### Hierarchical retrieval pattern

A production system would usually maintain separate indexes or metadata for summaries and leaf chunks.

The important idea is:

**retrieve the relevant area first → retrieve detailed evidence inside it.**

In [ ]:
def hierarchical_retrieval(question):
    # Pseudocode:
    #
    # 1. Search the summary index.
    # 2. Identify relevant documents/sections.
    # 3. Restrict detailed vector retrieval to those areas.
    # 4. Re-rank the detailed chunks.
    #
    # This reduces the search space and preserves document structure.

    raise NotImplementedError(
        "Connect this pattern to your summary and chunk indexes."
    )

# 14. Graph RAG

Vector search asks:

> Which text is semantically similar?

Graph RAG also models:

> Which entities are connected, and what information is connected through those relationships?

Example:

```text
Person → works_at → Company
Person → studied_at → University
Company → won → Award
```

Graph RAG is especially useful for multi-hop questions involving relationships across documents.

In [25]:
import networkx as nx

graph = nx.MultiDiGraph()


class Relation(BaseModel):
    source: str
    relation: str
    target: str


class Relations(BaseModel):
    relations: list[Relation]


def extract_relations(text):
    prompt = f"""
Extract important factual relationships from this text.

Represent each as:
source -> relation -> target

Only include relationships directly supported by the text.

Text:
{text}
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=Relations,
    )

    return Relations.model_validate_json(
        response.choices[0].message.content
    ).relations


def add_relations_to_graph(relations, source_chunk_id=None):
    for rel in relations:
        graph.add_edge(
            rel.source,
            rel.target,
            relation=rel.relation,
            source_chunk=source_chunk_id,
        )


# Example:
relations = extract_relations(llm_chunks[0]["page_content"])
add_relations_to_graph(relations, source_chunk_id=0)

### Graph-assisted retrieval

A practical hybrid architecture is:

```text
Vector retrieval
      ↓
Relevant entities
      ↓
Graph neighborhood expansion
      ↓
Linked source chunks
      ↓
Re-ranking
```

The graph does not have to replace vector search. It can add relationship-aware context.

In [28]:
def graph_neighbors(entity, hops=1):
    if entity not in graph:
        return set()

    current = {entity}
    visited = {entity}

    for _ in range(hops):
        next_nodes = set()

        for node in current:
            next_nodes.update(graph.successors(node))
            next_nodes.update(graph.predecessors(node))

        next_nodes -= visited
        visited.update(next_nodes)
        current = next_nodes

    return visited


# Example:
print(graph_neighbors("Rellm", hops=2))

{'decision-making processes', 'decision-making', 'Insurellm', 'platform', 'operational efficiency', 'operational efficiencies', 'risk management', 'AI-Powered Enterprise Reinsurance Solution', 'insurers', 'product', 'Rellm'}


# 15. Agentic RAG

Normal RAG is a fixed pipeline:

```text
question → retrieve → answer
```

Agentic RAG lets an agent decide **what to do next**.

Possible tools:
- vector retrieval
- graph lookup
- SQL
- web search
- calculator
- memory

The agent is useful when a question cannot reliably be answered with one retrieval operation.

In [29]:
class ToolDecision(BaseModel):
    tool: str = Field(
        description="One of: vector_search, graph_search, sql, final"
    )
    query: str = Field(
        description="Input for the selected tool"
    )


def agent_decide(question, history, available_tools):
    prompt = f"""
You are the retrieval planner for a RAG system.

Question:
{question}

History:
{history}

Available tools:
{available_tools}

Choose the single best next tool.

vector_search = semantic document retrieval
graph_search = relationship lookup
sql = structured data lookup
final = enough evidence has been collected

Return only the tool decision.
"""

    response = completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=ToolDecision,
    )

    return ToolDecision.model_validate_json(
        response.choices[0].message.content
    )

### Simple agent loop

The architecture is:

```text
Question
   ↓
Agent chooses tool
   ↓
Tool runs
   ↓
Observation
   ↓
Agent chooses again
   ↓
Final answer
```

Keep strict limits on steps and tools in real systems.

In [32]:
def run_agent(question, collection, max_steps=3):
    history = []
    observations = []

    tools = """
vector_search: semantic document retrieval
graph_search: relationship lookup
sql: structured database lookup
final: answer using gathered evidence
"""

    for step in range(max_steps):
        decision = agent_decide(
            question,
            history + observations,
            tools,
        )

        if decision.tool == "vector_search":
            print("Doing vector search")
            result = fetch_context_unranked(
                decision.query,
                collection,
                k=10,
            )

            observation = "\n\n".join(
                item["page_content"] for item in result
            )

        elif decision.tool == "graph_search":
            print("Doing graph search")
            related = graph_neighbors(
                decision.query,
                hops=2,
            )
            observation = str(list(related))

        elif decision.tool == "sql":
            print("Doing SQL query")
            observation = (
                "SQL tool placeholder: execute a validated "
                "read-only SQL query here."
            )

        elif decision.tool == "final":
            break

        else:
            observation = "Unknown tool."

        observations.append(
            f"Step {step + 1}: {observation}"
        )

    return observations


# Example:
observations = run_agent(
     "About rellm",
    collection,
 )
observations

Doing vector search
Doing vector search


["Step 1: Rellm Features: AI Analytics, Integrations, Risk Assessment, Dashboard, Compliance, and Portals\n\nRellm offers AI-driven analytics for predictive insights, seamless integrations with existing systems, a comprehensive risk assessment module, a customizable dashboard, regulatory compliance tools, and dedicated portals for clients and brokers.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integrations\nRellm's architecture is designed for effortless integration with existing systems. Whether it's policy management, claims processing, or financial reporting, Rellm connects seamlessly with diverse data sources to create a unified ecosystem.\n\n### Risk Assessment Module\nThe comprehensive risk assessment module within Rel

# 16. Final answer generation

Once retrieval is good, generation should mainly be responsible for **reasoning over the selected evidence**.

Do not pass a huge noisy context just because the model can accept it. Retrieval and re-ranking should do the context selection.

In [ ]:
def answer_question(question, collection, history=None):
    history = history or []

    chunks = advanced_retrieval(
        question,
        collection,
    )

    messages = make_rag_messages(
        question,
        history,
        chunks,
    )

    response = completion(
        model=LLM_MODEL,
        messages=messages,
    )

    return response.choices[0].message.content, chunks


# Example:
# answer, context = answer_question(
#     "Who won the IIOTY award?",
#     collection,
# )
# print(answer)

# 17. Professional hybrid RAG architecture

The techniques can be combined into a pipeline like this:

```text
                         USER QUESTION
                              ↓
                     Query Rewriting
                              ↓
                      Query Expansion
                              ↓
                ┌─────────────┴─────────────┐
                ↓                           ↓
          Vector Search                Graph Search
                ↓                           ↓
                └─────────────┬─────────────┘
                              ↓
                     Merge Candidates
                              ↓
                          Re-rank
                              ↓
                       Final Context K
                              ↓
                     Improved Prompt
                              ↓
                             LLM
                              ↓
                           ANSWER
```

Hierarchical retrieval can be used before detailed retrieval, and an agent can decide which path to use.

In [ ]:
def hybrid_rag(question, collection):
    # 1. Rewrite
    rewritten = rewrite_query(question)

    # 2. Retrieve original + rewritten query
    result_sets = [
        fetch_context_unranked(question, collection, k=10),
        fetch_context_unranked(rewritten, collection, k=10),
    ]

    # 3. Merge
    candidates = merge_unique_chunks(result_sets)

    # 4. Re-rank
    ranked = rerank(question, candidates)

    # 5. Final context
    return ranked[:FINAL_K]

# 18. Evaluation

Advanced RAG techniques should not be selected by intuition.

Use the **same fixed test set** after every change.

Retrieval metrics:
- **MRR** — how high the first relevant result appears
- **nDCG** — how well relevant results are ranked overall
- **Recall@K / keyword coverage** — whether relevant information appears in the top K
- **Precision@K** — how much of top K is relevant

Answer metrics:
- **Accuracy**
- **Completeness**
- **Relevance**

The existing Day 5 evaluation approach separates retrieval quality from answer quality. That is important: a good answer can sometimes hide weak retrieval, and a strong retriever can still have a poor answer prompt.

In [ ]:
# Keep an experiment log.

experiments = [
    {
        "strategy": "MiniLM + recursive chunks",
        "mrr": None,
        "ndcg": None,
        "coverage": None,
        "accuracy": None,
    },
    {
        "strategy": "BGE-small + recursive chunks",
        "mrr": None,
        "ndcg": None,
        "coverage": None,
        "accuracy": None,
    },
    {
        "strategy": "BGE + query rewriting",
        "mrr": None,
        "ndcg": None,
        "coverage": None,
        "accuracy": None,
    },
    {
        "strategy": "BGE + rewrite + rerank",
        "mrr": None,
        "ndcg": None,
        "coverage": None,
        "accuracy": None,
    },
]

experiments

# 19. Recommended learning order

Do not implement everything at once.

### Level 1 — Retrieval fundamentals

1. Baseline chunking
2. Embedding model experiments
3. Retrieval `k`
4. Evaluation

### Level 2 — High-value improvements

5. Query rewriting
6. Re-ranking
7. Query expansion
8. Better prompts

### Level 3 — Advanced architecture

9. Document pre-processing
10. Hierarchical RAG
11. Graph RAG
12. Agentic RAG

This lets you understand what each component contributes before combining them.

# 20. Final mental model

A RAG system is not just:

```text
Embedding → Vector DB → LLM
```

A professional RAG system is an information-retrieval pipeline:

```text
                 OFFLINE / INDEXING

Documents
   ↓
Pre-processing
   ↓
Chunking
   ↓
Metadata / summaries / graph
   ↓
Embedding
   ↓
Vector + other indexes


                 ONLINE / QUERYING

User question
   ↓
Query rewriting / expansion
   ↓
Retrieval
   ↓
Candidate merging
   ↓
Re-ranking
   ↓
Context selection
   ↓
Prompt + history
   ↓
LLM
   ↓
Answer


                 EVALUATION

Same test set
   ↓
Retrieval metrics
   ↓
Answer metrics
   ↓
Compare strategies
   ↓
Keep what actually improves performance
```

The professional skill is not memorizing every RAG technique. It is being able to **measure a baseline, change one part of the pipeline, evaluate it, and understand why the result changed**.